# Bayesian optimization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Use a probabilistic surrogate to decide the next expensive evaluation
Bayesian optimization is for objectives where every evaluation is costly. These five examples show a Gaussian-process posterior mean, uncertainty shrinking near observations, expected improvement, exploration versus exploitation through UCB, and a sequential next-point choice.

In [ ]:
# 1) simple RBF surrogate posterior mean
X=np.array([[0.],[1.],[2.]]); y=np.array([0.,1.,0.]); grid=np.linspace(0,2,100)[:,None]; k=lambda A,B: np.exp(-0.5*((A-B.T)**2)/0.3**2)
K=k(X,X)+1e-6*np.eye(3); Ks=k(grid,X); mean=Ks@np.linalg.solve(K,y)
plt.figure(figsize=(5,3)); plt.plot(grid[:,0],mean); plt.scatter(X[:,0],y,c='r'); plt.title('surrogate interpolates observations')
assert abs(mean[np.argmin(abs(grid[:,0]-1))]-1)<0.02

In [ ]:
# 2) posterior uncertainty is smallest near data
var=1-np.sum(Ks@np.linalg.inv(K)*Ks,axis=1); plt.figure(figsize=(5,3)); plt.plot(grid[:,0],np.sqrt(np.maximum(var,0))); plt.scatter(X[:,0],np.zeros(3),c='r'); plt.title('uncertainty dips at observed points')
assert var[np.argmin(abs(grid[:,0]-1))]<var[np.argmin(abs(grid[:,0]-0.5))]

In [ ]:
# 3) expected improvement balances mean and uncertainty
best=y.max(); sigma=np.sqrt(np.maximum(var,1e-12)); z=(mean-best)/sigma; ei=(mean-best)*0.5*(1+np.vectorize(math.erf)(z/np.sqrt(2)))+sigma*(1/np.sqrt(2*np.pi))*np.exp(-0.5*z*z)
plt.figure(figsize=(5,3)); plt.plot(grid[:,0],ei); plt.title('expected improvement')
assert ei.max()>0

In [ ]:
# 4) UCB turns an exploration knob
ucb1=mean+0.5*sigma; ucb2=mean+2.0*sigma; plt.figure(figsize=(5,3)); plt.plot(grid[:,0],ucb1,label='kappa=.5'); plt.plot(grid[:,0],ucb2,label='kappa=2'); plt.legend(); plt.title('larger kappa values uncertainty')
assert ucb2[np.argmax(sigma)]>ucb1[np.argmax(sigma)]

In [ ]:
# 5) next point maximizes acquisition, not raw mean
acq=mean+1.5*sigma; xnext=grid[np.argmax(acq),0]
plt.figure(figsize=(5,3)); plt.plot(grid[:,0],acq); plt.axvline(xnext,c='r',ls='--'); plt.title(f'next x={xnext:.2f}')
assert 0<=xnext<=2 and abs(xnext-1)>0.05